In [1]:
from pathlib import Path
from datetime import datetime, timezone
import re, json, math
import pandas as pd
from collections import Counter

RAW_DIR = Path("/home/jfrudzinski/work/Discord/Analytics/nomad_discord_data_2025-09-08_15-46-23/issues")   # the raw JSON threads by joe
OUT_DIR = Path("/home/jfrudzinski/work/Discord/nomad-bot-rag-docs-discord/external/discord")
OUT_DIR.mkdir(parents=True, exist_ok=True)

def slugify(s: str) -> str:
    s = (s or "").strip().lower()
    s = re.sub(r"[^a-z0-9]+", "-", s)
    s = re.sub(r"-+", "-", s).strip("-")
    return s or "untitled"

def to_utc_iso(ts: str) -> str:
    if not ts:
        return ""
    try:
        dt = datetime.fromisoformat(ts)
        return dt.astimezone(timezone.utc).replace(tzinfo=timezone.utc).isoformat().replace("+00:00", "Z")
    except Exception:
        return ts

json_files = sorted(RAW_DIR.glob("*.json"))
len(json_files), [p.name for p in json_files[:5]]


(303,
 ['NOMAD - issues - "This entry does not exist." upon publishing [1275052785494392914].json',
  'NOMAD - issues - # not escaped in API calls [1301134820063313992].json',
  'NOMAD - issues - .hdf5 files (own made .h5, .h5iona, ...) for publication on central nomad [1374638026356949032].json',
  'NOMAD - issues - 1.3.14? [1338856302100873267].json',
  'NOMAD - issues - 2D array of strings [1250003438503333938].json'])

In [ ]:
import re

# --- Keep code blocks intact while cleaning surrounding text ---
CODEBLOCK_RE = re.compile(r"```[\s\S]*?```", re.M)

# Lines that are *only* a sign-off/ack
SIGNOFF_LINE_RE = re.compile(
    r"^\s*(thanks|thank you|many thanks|cheers|best(?: regards)?|kind regards|br|rgds|thx)[\.,!]*\s*$",
    re.I,
)

# Leading courtesy/openers to strip from the *start* of a message
LEADING_FLUFF = [
    r"(?:hi|hey|hello|dear(?: all)?|good (?:morning|afternoon|evening))[,!.\s-]*",
    r"(?:thanks|thank you|many thanks|thx)[,!.\s-]*",
    r"(?:sorry(?: about that| for the late reply)?)[:,!.\s-]*",
    r"(?:quick question)[:\-\s]*",
    # common “thinking/wondering” preambles
    r"(?:i was (?:thinking|wondering|trying) about this)[:,\-\s]*",
    r"(?:i was (?:thinking|wondering|trying))[:,\-\s]*",
]

LEADING_FLUFF_RE = re.compile(r"^(?:" + r"|".join(LEADING_FLUFF) + r")", re.I)

# Redact any @mention leftovers
MENTION_RE = re.compile(r"@\S+")

# Collapse repeated punctuation/whitespace
MULTI_PUNCT_RE = re.compile(r"([!?.,])\1{2,}")   # "!!!" -> "!!", "...." -> "..."
WS_RE          = re.compile(r"\s+")

# (Optional) strip simple ASCII emoji/emoticons
ASCII_EMOJI_RE = re.compile(r"(:-\)|:-\(|;-\)|:\)|:\(|;-\))")


def trim_conversational_fluff(text: str) -> str:
    """
    Make Discord messages read more like documentation snippets:
    - keep code blocks verbatim
    - remove greeting/thanks/apology openers (start-of-message)
    - drop lines that are pure sign-offs/acks
    - redact any @mentions that slipped through
    - collapse excessive punctuation/whitespace
    """
    if not text:
        return ""

    # 1) Split out code blocks to avoid touching them
    parts = CODEBLOCK_RE.split(text)  # keeps delimiters if pattern has groups? (no groups -> split removes blocks)
    # Use finditer to preserve blocks exactly:
    rebuilt = []
    last = 0
    for m in CODEBLOCK_RE.finditer(text):
        pre = text[last:m.start()]
        code = text[m.start():m.end()]
        rebuilt.append(_clean_noncode(pre))
        rebuilt.append(code)  # keep code block verbatim
        last = m.end()
    rebuilt.append(_clean_noncode(text[last:]))

    cleaned = "".join(rebuilt).strip()
    return cleaned


def _clean_noncode(chunk: str) -> str:
    if not chunk:
        return ""

    # Remove sign-off lines and trim leading fluff on the first substantive line
    lines = [l for l in chunk.splitlines()]
    # drop pure sign-off/ack lines
    lines = [l for l in lines if not SIGNOFF_LINE_RE.match(l)]

    # Strip leading fluff repeatedly on the first non-empty line
    for i, line in enumerate(lines):
        s = line.strip()
        if not s:
            continue
        # redact any @mentions (after your rewrite_mentions pass)
        s = MENTION_RE.sub("[REDACTED_MENTION]", s)
        # strip common greeting/thanks/apology preambles at *start*
        prev = None
        while prev != s:
            prev = s
            s = LEADING_FLUFF_RE.sub("", s).lstrip()
        # light cleanup
        s = ASCII_EMOJI_RE.sub("", s)
        s = MULTI_PUNCT_RE.sub(r"\1\1", s)  # cap repeats to length 2
        lines[i] = s
        break  # only the first substantive line gets opener stripping

    # Collapse whitespace on all lines; keep line breaks to preserve structure
    lines = [WS_RE.sub(" ", l).strip() for l in lines]
    # remove empty lines that may result
    lines = [l for l in lines if l]

    return ("\n".join(lines) + ("\n" if chunk.endswith("\n") else ""))


In [ ]:
def normalize_name(name: str) -> str:
    s = (name or "").lstrip("@").strip()
    s = re.sub(r"\s+", "_", s)
    s = re.sub(r"[\u200b-\u200f\u202a-\u202e]", "", s)
    return s

def rewrite_mentions(text: str, mentions: list) -> str:
    if not text:
        return ""
    out = text

    # Step 1: replace known mentions with normalized form
    reps = []
    for m in mentions or []:
        nick = m.get("nickname") or m.get("name")
        name = m.get("name")
        cands = []
        if nick:
            cands.append(nick)
        if name and name != nick:
            cands.append(name)
        uniq = sorted(set(cands), key=lambda s: len(s), reverse=True)
        norm = normalize_name(nick or name or "")
        if not norm:
            continue
        for cand in uniq:
            reps.append((r"@" + re.escape(cand), f"@{norm}"))

    seen, ordered = set(), []
    for pat, repl in reps:
        if pat not in seen:
            seen.add(pat)
            ordered.append((pat, repl))
    for pat, repl in ordered:
        out = re.sub(pat, repl, out)

    # Step 2: redact *any remaining* @mentions
    out = re.sub(r"@\S+", "[REDACTED_MENTION]", out)

    return out


def redact_pii(text: str) -> str:
    # emails
    text = re.sub(r"[A-Za-z0-9._%+\-]+@[A-Za-z0-9.\-]+\.[A-Za-z]{2,}", "[REDACTED_EMAIL]", text)
    # rough phones
    text = re.sub(r"\b(?:\+?\d{1,3}[-.\s]?)?(?:\(?\d{2,4}\)?[-.\s]?){2,4}\d{2,4}\b", "[REDACTED_PHONE]", text)
    return text

def load_thread_as_text(path: Path):
    data = json.loads(path.read_text(encoding="utf-8"))
    guild = data.get("guild", {}) or {}
    channel = data.get("channel", {}) or {}
    msgs = data.get("messages", []) or []

    title   = channel.get("name") or path.stem
    section = channel.get("category") or guild.get("name")
    first_ts = None

    texts = []
    for m in msgs:
        t = (m.get("content") or "").strip()
        if not t:
            continue
        t = rewrite_mentions(t, m.get("mentions") or [])
        t = redact_pii(t)
        t = trim_conversational_fluff(t)  # 👈 add this
        if len(t) < 3:
            continue
        texts.append(t)
        if not first_ts:
            first_ts = to_utc_iso(m.get("timestamp"))

    full_text = "\n".join(texts)
    return {
        "source": "discord",
        "title": title,
        "section": section,
        "text": full_text,
        "url": None,
        "timestamp": first_ts or "",
    }


threads = [load_thread_as_text(p) for p in json_files]
len(threads), threads[0]["title"], len(threads[0]["text"])


(303, '"This entry does not exist." upon publishing', 3496)

In [3]:
import json, hashlib
from pathlib import Path

def thread_id(thread: dict) -> str:
    """Stable id from title + first timestamp + short hash of text."""
    base = f"{thread.get('title','')}|{thread.get('timestamp','')}"
    h = hashlib.sha1(thread.get('text','').encode('utf-8')).hexdigest()[:10]
    return f"{base}|{h}"

def save_threads_jsonl(threads: list[dict], out_path: Path):
    out_path.parent.mkdir(parents=True, exist_ok=True)
    # Atomic-ish write
    tmp = out_path.with_suffix(out_path.suffix + ".tmp")
    with tmp.open("w", encoding="utf-8") as f:
        for th in threads:
            rec = {
                "id": thread_id(th),
                "source": th.get("source","discord"),
                "title": th.get("title",""),
                "section": th.get("section",""),
                "text": th.get("text",""),
                "url": th.get("url"),
                "timestamp": th.get("timestamp",""),
                "char_count": len(th.get("text","")),
            }
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")
    tmp.replace(out_path)
    return out_path

# usage
out_jsonl = Path("../external/discord/stripped/issues.jsonl")
save_threads_jsonl(threads, out_jsonl)
print(f"Saved {len(threads)} threads to {out_jsonl}")


Saved 303 threads to ../external/discord/stripped/issues.jsonl


In [4]:
# def tokenize_words(text: str):
#     return re.findall(r"[A-Za-zÄÖÜäöüß0-9]+", text or "")

# def split_sentences(text: str):
#     parts = re.split(r"(?<=[.!?])\s+(?=[A-ZÄÖÜ])", (text or "").strip())
#     return [p.strip() for p in parts if p.strip()] or [text.strip()]

# def chunk_fixed(text: str, max_words: int = 400, overlap: int = 50):
#     toks = tokenize_words(text)
#     if not toks:
#         return []
#     chunks, i, N = [], 0, len(toks)
#     while i < N:
#         j = min(i + max_words, N)
#         chunks.append(" ".join(toks[i:j]))
#         if j == N:
#             break
#         i = max(0, j - overlap)
#     return chunks

# def chunk_sentence(text: str, target_words: int = 350):
#     sents = split_sentences(text)
#     chunks, buf, count = [], [], 0
#     for s in sents:
#         n = len(tokenize_words(s))
#         if count + n > target_words and buf:
#             chunks.append(" ".join(buf)); buf, count = [], 0
#         buf.append(s); count += n
#     if buf:
#         chunks.append(" ".join(buf))
#     return chunks

# def bow_vector(text: str) -> Counter:
#     return Counter(tokenize_words((text or "").lower()))

# def cosine_sim(a: Counter, b: Counter) -> float:
#     if not a or not b:
#         return 0.0
#     inter = set(a) & set(b)
#     num = sum(a[t]*b[t] for t in inter)
#     da = math.sqrt(sum(v*v for v in a.values()))
#     db = math.sqrt(sum(v*v for v in b.values()))
#     if da == 0 or db == 0:
#         return 0.0
#     return num / (da * db)

# def chunk_semantic(text: str, target_words: int = 350, min_sim: float = 0.25):
#     sents = split_sentences(text)
#     if not sents:
#         return []
#     chunks, buf, centroid, count = [], [], Counter(), 0
#     for s in sents:
#         sv = bow_vector(s)
#         sim = cosine_sim(centroid, sv) if centroid else 1.0
#         n = len(tokenize_words(s))
#         if (sim < min_sim and buf) or (count + n > target_words and buf):
#             chunks.append(" ".join(buf)); buf, centroid, count = [], Counter(), 0
#         buf.append(s); centroid.update(sv); count += n
#     if buf:
#         chunks.append(" ".join(buf))
#     return chunks


In [4]:
FIX_MAX, FIX_OVERLAP   = 400, 50
SENT_TARGET            = 350
SEM_TARGET, SEM_MINSIM = 350, 0.25

out_fixed    = OUT_DIR / "nomad_discord.fixed.jsonl"
out_sentence = OUT_DIR / "nomad_discord.sentence.jsonl"
out_semantic = OUT_DIR / "nomad_discord.semantic.jsonl"

def write_jsonl(rows, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

fixed_rows, sentence_rows, semantic_rows = [], [], []

for thr in threads:
    title   = thr["title"]
    section = thr["section"]
    url     = thr["url"]
    ts      = thr["timestamp"]
    text    = thr["text"]

    # fixed
    for i, ch in enumerate(chunk_fixed(text, max_words=FIX_MAX, overlap=FIX_OVERLAP)):
        fixed_rows.append({
            "id": f"discord:{slugify(title)}:{slugify(section)}:fixed:{i}",
            "source": "discord",
            "title": title,
            "section": section,
            "text": ch,
            "url": url,
            "timestamp": ts
        })
    # sentence
    for i, ch in enumerate(chunk_sentence(text, target_words=SENT_TARGET)):
        sentence_rows.append({
            "id": f"discord:{slugify(title)}:{slugify(section)}:sentence:{i}",
            "source": "discord",
            "title": title,
            "section": section,
            "text": ch,
            "url": url,
            "timestamp": ts
        })
    # semantic
    for i, ch in enumerate(chunk_semantic(text, target_words=SEM_TARGET, min_sim=SEM_MINSIM)):
        semantic_rows.append({
            "id": f"discord:{slugify(title)}:{slugify(section)}:semantic:{i}",
            "source": "discord",
            "title": title,
            "section": section,
            "text": ch,
            "url": url,
            "timestamp": ts
        })

write_jsonl(fixed_rows, out_fixed)
write_jsonl(sentence_rows, out_sentence)
write_jsonl(semantic_rows, out_semantic)

len(fixed_rows), len(sentence_rows), len(semantic_rows), out_fixed.as_posix(), out_sentence.as_posix(), out_semantic.as_posix()


(545,
 599,
 3418,
 '/home/ebb/llm-hackathon/data/discord/nomad_discord.fixed.jsonl',
 '/home/ebb/llm-hackathon/data/discord/nomad_discord.sentence.jsonl',
 '/home/ebb/llm-hackathon/data/discord/nomad_discord.semantic.jsonl')

In [5]:
# size & a peek
for p in [out_fixed, out_sentence, out_semantic]:
    n = sum(1 for _ in p.open("r", encoding="utf-8"))
    print(p.name, "lines:", n)

# load a few rows
sample = []
with out_sentence.open("r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i >= 5: break
        sample.append(json.loads(line))
pd.DataFrame([{
    "id": r["id"], "title": r["title"], "section": r["section"], "chars": len(r["text"]), "preview": r["text"][:200]
} for r in sample])


nomad_discord.fixed.jsonl lines: 545
nomad_discord.sentence.jsonl lines: 599
nomad_discord.semantic.jsonl lines: 3418


,id,title,section,chars,preview
0,discord:this-entry-does-not-exist-upon-publish...,"""This entry does not exist."" upon publishing",issues,1861,"Hi everyone, here's an issue I'm seeing when p..."
1,discord:this-entry-does-not-exist-upon-publish...,"""This entry does not exist."" upon publishing",issues,1621,"If you encounter this for other of your files,..."
2,discord:not-escaped-in-api-calls:issues:senten...,# not escaped in API calls,issues,436,"Hey all,\nI have file names including # and i ..."
3,discord:hdf5-files-own-made-h5-h5iona-for-publ...,".hdf5 files (own made .h5, .h5iona, ...) for p...",issues,1602,I have to upload all sorts of .hdf5 files (own...
4,discord:hdf5-files-own-made-h5-h5iona-for-publ...,".hdf5 files (own made .h5, .h5iona, ...) for p...",issues,2472,Is there any doc about that?\nin order to make...
